In [362]:
import numpy as np
import pandas as pd

In [363]:
df=pd.read_csv("Superstore sales dataset.csv")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,8/11/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,8/11/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,12/6/2016,16/6/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,11/10/2015,18/10/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,11/10/2015,18/10/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


same id for many products

In [364]:
problem1 = df.groupby("Product ID")["Product Name"].nunique()
problem1 = problem1[problem1 > 1]
print(problem1)

Product ID
FUR-BO-10002213    2
FUR-CH-10001146    2
FUR-FU-10001473    2
FUR-FU-10004017    2
FUR-FU-10004091    2
FUR-FU-10004270    2
FUR-FU-10004848    2
FUR-FU-10004864    2
OFF-AP-10000576    2
OFF-AR-10001149    2
OFF-BI-10002026    2
OFF-BI-10004632    2
OFF-BI-10004654    2
OFF-PA-10000357    2
OFF-PA-10000477    2
OFF-PA-10000659    2
OFF-PA-10001166    2
OFF-PA-10001970    2
OFF-PA-10002195    2
OFF-PA-10002377    2
OFF-PA-10003022    2
OFF-ST-10001228    2
OFF-ST-10004950    2
TEC-AC-10002049    2
TEC-AC-10002550    2
TEC-AC-10003832    2
TEC-MA-10001148    2
TEC-PH-10001530    2
TEC-PH-10001795    2
TEC-PH-10002200    2
TEC-PH-10002310    2
TEC-PH-10004531    2
Name: Product Name, dtype: int64


same product with many ids


In [365]:
problem2 = df.groupby("Product Name")["Product ID"].nunique()
problem2 = problem2[problem2 > 1]
print(problem2)

Product Name
#10- 4 1/8" x 9 1/2" Recycled Envelopes           2
Avery Non-Stick Binders                           2
Easy-staple paper                                 8
Eldon Wave Desk Accessories                       2
KI Adjustable-Height Table                        2
Okidata C610n Printer                             2
Peel & Seel Recycled Catalog Envelopes, Brown     2
Prang Drawing Pencil Set                          2
Staple envelope                                   9
Staple holder                                     3
Staple magnet                                     2
Staple remover                                    3
Staple-based wall hangings                        2
Staples                                          10
Staples in misc. colors                           7
Storex Dura Pro Binders                           2
Name: Product ID, dtype: int64


In [366]:
print("number of same id with many products =", len(problem1))
print("number of same product with many ids =", len(problem2))

number of same id with many products = 32
number of same product with many ids = 16


delete column and duplicated

In [367]:
df = df.drop_duplicates()
df = df.drop(columns=["Postal Code", "Country", "Row ID"])


edit city

In [368]:
df["City"] = df["City"].str.strip().str.title()
df["Product Name"] = df["Product Name"].str.strip().str.title()
df.columns = df.columns.str.strip()


make custmoer table


In [369]:
customers = df[["Customer Name", "Customer ID", "Segment"]]
customers = customers.drop_duplicates().reset_index(drop=True)

make location table and merge it


In [370]:
location = df[["City", "State", "Region"]]
location = location.drop_duplicates().reset_index(drop=True)
location["Location ID"] = location.index + 1
df = df.merge(location, on=["City", "State", "Region"], how="left")



make order table

In [371]:
orders = df[["Order ID", "Order Date", "Ship Date", "Ship Mode","Location ID", "Customer ID"]]
orders = orders.drop_duplicates().reset_index(drop=True)

change date

In [372]:
orders["Order Date"] = pd.to_datetime(orders["Order Date"], errors="coerce")
orders["Ship Date"] = pd.to_datetime(orders["Ship Date"], errors="coerce")

calculate shippind day

In [373]:
orders["Shipping Day"] = (orders["Ship Date"] - orders["Order Date"]).dt.days

edit product

In [374]:
df["Product Name"] = df["Product Name"].str.strip().str.title()
products = df.groupby("Product Name", as_index=False).agg({
    "Product ID": "first"
})

solve same id with multi products problem

In [375]:
problem_ids = df.groupby("Product ID")["Product Name"].nunique()
problem_ids = problem_ids[problem_ids > 1].index
df_problem = df[df["Product ID"].isin(problem_ids)].copy()
df_problem = df_problem.sort_values(["Product ID", "Product Name"])
df_problem["rank"] = df_problem.groupby("Product ID").cumcount()
df_problem["Product ID"] = df_problem.apply(
    lambda row: str(row["Product ID"]) if row["rank"] == 0
    else str(row["Product ID"]) + "0" * row["rank"],
    axis=1
)

df.update(df_problem)


merge product table

In [376]:
df = df.merge(
    products[["Product Name", "Product ID"]],
    on="Product Name",
    how="left"
)
if "Product ID_y" in df.columns:
    df["Product ID"] = df["Product ID_y"]
elif "Product ID_x" in df.columns:
    df["Product ID"] = df["Product ID_x"]

calclate profit margin , cost and make order deatils table

In [377]:

order_details = df[[
    "Order ID",
    "Product ID",
    "Sales",
    "Quantity",
    "Discount",
    "Profit"
]].copy()
order_details["Profit Margin"] = order_details["Profit"] / order_details["Sales"]
order_details["Cost"] = order_details["Sales"] - order_details["Profit"]
order_details = order_details.drop_duplicates(
    subset=["Order ID", "Product ID"]
)



solve same id with many products name problem

In [378]:
df.groupby("Product ID")["Product Name"].nunique()

,Product Name
Product ID,
FUR-BO-10000112,1
FUR-BO-10000330,1
FUR-BO-10000362,1
FUR-BO-10000468,1
FUR-BO-10000711,1
...,...
TEC-PH-10004912,1
TEC-PH-10004922,1
TEC-PH-10004924,1


In [379]:
products.shape[0]

1850

In [380]:
order_details.shape

(9986, 8)

In [381]:
orders.shape

(5009, 7)

In [382]:
customers.shape

(793, 3)

In [383]:
location.shape

(604, 4)

In [384]:
df.shape

(9994, 21)

customer analysis

total num of customers

In [385]:
customers["Customer ID"].nunique()

793

total sales

In [386]:
total_sales = df.drop_duplicates(subset=[
    "Order ID",
    "Product ID",
    "Sales",
    "Quantity",
    "Discount",
    "Profit"
])["Sales"].sum()

print(round(total_sales, 2))

2296919.49


top 10 customers by sales

In [387]:
top_customers_sales = (
    df.groupby("Customer ID")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_customers_sales = top_customers_sales.merge(customers, on="Customer ID")
top_customers_sales = top_customers_sales.drop(columns=['Segment'])
top_customers_sales

,Customer ID,Sales,Customer Name
0,SM-20320,25043.050,Sean Miller
1,TC-20980,19052.218,Tamara Chand
2,RB-19360,15117.339,Raymond Buch
3,TA-21385,14595.620,Tom Ashbrook
4,AB-10105,14473.571,Adrian Barton
5,KL-16645,14175.229,Ken Lonsdale
6,SC-20095,14142.334,Sanjit Chand
7,HL-15040,12873.298,Hunter Lopez
8,SE-20110,12209.438,Sanjit Engle
9,CC-12370,12129.072,Christopher Conant


top 10 customers by profit

In [388]:
top_customers_profit = (
    df.groupby("Customer ID")["Profit"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

top_customers_profit = top_customers_profit.merge(customers, on="Customer ID")
top_customers_profit = top_customers_profit.drop(columns=['Segment'])

top_customers_profit

,Customer ID,Profit,Customer Name
0,TC-20980,8981.3239,Tamara Chand
1,RB-19360,6976.0959,Raymond Buch
2,SC-20095,5757.4119,Sanjit Chand
3,HL-15040,5622.4292,Hunter Lopez
4,AB-10105,5444.8055,Adrian Barton
5,TA-21385,4703.7883,Tom Ashbrook
6,CM-12385,3899.8904,Christopher Martinez
7,KD-16495,3038.6254,Keith Dawkins
8,AR-10540,2884.6208,Andy Reiter
9,DR-12940,2869.0760,Daniel Raglin


most frequent num

In [389]:
most_frequent_customers = (
    df.groupby('Customer Name')['Order ID']
    .nunique()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
most_frequent_customers.columns = ['Customer Name', 'Order Count']
most_frequent_customers

,Customer Name,Order Count
0,Emily Phan,17
1,Sally Hughsby,13
2,Noel Staavos,13
3,Patrick Gardner,13
4,Zuschuss Carroll,13
5,Joel Eaton,13
6,Erin Ashbrook,13
7,Chloris Kastensmidt,13
8,Suzanne McNair,12
9,Rick Bensley,12


	Customers Buying a Lot but Lose profit

In [390]:
customer_analysis = (
    df.groupby('Customer Name')
    .agg({
        'Sales': 'sum',
        'Profit': 'sum'
    })
    .reset_index()
)
#filter customers that lose
loss_customers = customer_analysis[customer_analysis['Profit'] < 0]
#arrange it eith best sales
loss_customers = loss_customers.sort_values(by='Sales', ascending=False).head(10)
loss_customers

,Customer Name,Sales,Profit
686,Sean Miller,25043.050,-1980.7393
75,Becky Martin,11789.630,-1659.9581
307,Grant Thornton,9351.212,-4108.6589
606,Peter Fuller,9062.864,-614.2943
557,Natalie Fritzler,8322.826,-1695.9714
684,Sean Braxton,8057.891,-2082.7451
791,Zuschuss Carroll,8025.707,-1032.1490
397,Joseph Holt,7954.998,-644.6982
396,Joseph Airdo,6491.026,-819.4217
782,Victoria Wilson,6134.038,-874.6645


Segment Performance by Sales, Profit

In [423]:

segment_performance = (
    df.groupby('Segment')
    .agg({
        'Sales': 'sum',
        'Profit': 'sum'
    })
    .reset_index()
)
segment_performance

,Segment,Sales,Profit
0,Consumer,1.161401e+06,134119.2092
1,Corporate,7.061464e+05,91979.1340
2,Home Office,4.296531e+05,60298.6785


**product analysis**

total num of products

In [392]:
total_products = df['Product Name'].nunique()
total_products

1850

	Top 10 Products by Total Sales

In [393]:
top_products_sales = (
    df.groupby('Product Name')['Sales']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_products_sales.columns = ['Product Name', 'Total Sales']
top_products_sales

,Product Name,Total Sales
0,Canon Imageclass 2200 Advanced Copier,61599.824
1,Fellowes Pb500 Electric Punch Plastic Comb Bin...,27453.384
2,Cisco Telepresence System Ex90 Videoconferenci...,22638.480
3,Hon 5400 Series Task Chairs For Big And Tall,21870.576
4,Gbc Docubind Tl300 Electric Binding System,19823.479
5,Gbc Ibimaster 500 Manual Proclick Binding System,19024.500
6,Hewlett Packard Laserjet 3310 Copier,18839.686
7,Hp Designjet T520 Inkjet Large Format Printer ...,18374.895
8,Gbc Docubind P400 Electric Binding System,17965.068
9,High Speed Automatic Electric Letter Opener,17030.312


	Top 10 Products by Total Profit

In [428]:
top_products_profit = (
    df.groupby('Product Name')['Profit']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_products_profit.columns = ['Product Name', 'Total Profit']
top_products_profit

,Product Name,Total Profit
0,Canon Imageclass 2200 Advanced Copier,25199.9280
1,Fellowes Pb500 Electric Punch Plastic Comb Bin...,7753.0390
2,Hewlett Packard Laserjet 3310 Copier,6983.8836
3,Canon Pc1060 Personal Laser Copier,4570.9347
4,Hp Designjet T520 Inkjet Large Format Printer ...,4094.9766
5,Ativa V4110Mdd Micro-Cut Shredder,3772.9461
6,"3D Systems Cube Printer, 2Nd Generation, Magenta",3717.9714
7,Plantronics Savi W720 Multi-Device Wireless He...,3696.2820
8,Ibico Epk-21 Electric Binding System,3345.2823
9,Zebra Zm400 Thermal Label Printer,3343.5360


	Top 10 Most Sold Products by Quantity

In [429]:
top_products_quantity = (
    df.groupby('Product Name')['Quantity']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
top_products_quantity.columns = ['Product Name', 'Total Quantity']
top_products_quantity

,Product Name,Total Quantity
0,Staples,215
1,Staple Envelope,170
2,Easy-Staple Paper,150
3,Staples In Misc. Colors,86
4,Ki Adjustable-Height Table,74
...,...,...
1845,Global Enterprise Series Seating Low-Back Swiv...,1
1846,"Bush Saratoga Collection 5-Shelf Bookcase, Han...",1
1847,Boston 1900 Electric Pencil Sharpener,1
1848,Penpower Worldcard Pro Card Scanner,1


	Products Generating Loss

In [396]:
loss_products = (
    df.groupby('Product Name')['Profit']
    .sum()
    .reset_index()
)
# filter products that make negative profit
loss_products = loss_products[loss_products['Profit'] < 0]
#  arrange it with most negative profit
loss_products = loss_products.sort_values(by='Profit').reset_index(drop=True)
loss_products

,Product Name,Profit
0,Cubify Cubex 3D Printer Double Head Print,-8.879970e+03
1,Lexmark Mx611Dhe Monochrome Laser Printer,-4.589973e+03
2,Cubify Cubex 3D Printer Triple Head Print,-3.839990e+03
3,Chromcraft Bull-Nose Wood Oval Conference Tabl...,-2.876116e+03
4,Bush Advantage Collection Racetrack Conference...,-1.934398e+03
...,...,...
296,"Brites Rubber Bands, 1 1/2 Oz. Box",-5.148000e-01
297,Rubber Band Ball,-2.992000e-01
298,"Acco Presstex Data Binder With Storage Hooks, ...",-1.614000e-01
299,Premier Electric Letter Opener,-7.105427e-15


	Products with High Sales but Negative Profit

In [397]:
product_analysis = (
    df.groupby('Product Name')
    .agg({
        'Sales': 'sum',
        'Profit': 'sum'
    })
    .reset_index()
)
#filter products that make loss
high_sales_negative_profit = product_analysis[
    (product_analysis['Sales'] > product_analysis['Sales'].mean()) &
    (product_analysis['Profit'] < 0)
]
# arrange products with the hight sales and negative profit
high_sales_negative_profit = high_sales_negative_profit \
    .sort_values(by='Sales', ascending=False) \
    .reset_index(drop=True)
high_sales_negative_profit

,Product Name,Sales,Profit
0,Cisco Telepresence System Ex90 Videoconferenci...,22638.480,-1811.0784
1,Gbc Docubind P400 Electric Binding System,17965.068,-1878.1662
2,High Speed Automatic Electric Letter Opener,17030.312,-262.0048
3,Lexmark Mx611Dhe Monochrome Laser Printer,16829.901,-4589.9730
4,Martin Yale Chadless Opener Electric Letter Op...,16656.200,-1299.1836
...,...,...,...
118,Office Star - Ergonomically Designed Knee Chair,1311.876,-42.1096
119,Eldon Cleatmat Plus Chair Mats For High Pile C...,1304.128,-111.3280
120,"Hon 30"" X 60"" Table With Locking Drawer",1296.028,-177.3512
121,Tennsco Industrial Shelving,1281.442,-108.5802


Category Performance by Sales and Profit

In [430]:
category_performance = (
    df.groupby('Category')
    .agg({
        'Sales': 'sum',
        'Profit': 'sum'
    })
    .reset_index()
)
category_performance

,Category,Sales,Profit
0,Furniture,741999.7953,18451.2728
1,Office Supplies,719047.0320,122490.8008
2,Technology,836154.0330,145454.9481


	Sub_Categories by Sales and profit

In [431]:
sub_category_performance = (
    df.groupby('Sub-Category')
    .agg({
        'Sales': 'sum',
        'Profit': 'sum'
    })
    .reset_index()
)
sub_category_performance

,Sub-Category,Sales,Profit
0,Accessories,167380.3180,41936.6357
1,Appliances,107532.1610,18138.0054
2,Art,27118.7920,6527.7870
3,Binders,203412.7330,30221.7633
4,Bookcases,114879.9963,-3472.5560
5,Chairs,328449.1030,26590.1663
6,Copiers,149528.0300,55617.8249
7,Envelopes,16476.4020,6964.1767
8,Fasteners,3024.2800,949.5182
9,Furnishings,91705.1640,13059.1436


**order analysis**

Total Number of Orders

In [401]:
total_orders = df['Order ID'].nunique()
total_orders

5009

	Average Sales per Order

In [402]:
total_sales = df['Sales'].sum()
total_orders = df['Order ID'].nunique()
average_sales_per_order = total_sales / total_orders
average_sales_per_order

np.float64(458.61466566180883)

	Top 10 Sales Orders

In [403]:
top_orders = (
    df.groupby('Order ID')['Sales']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_orders.columns = ['Order ID', 'Total Sales']
top_orders

,Order ID,Total Sales
0,CA-2014-145317,23661.228
1,CA-2016-118689,18336.740
2,CA-2017-140151,14052.480
3,CA-2017-127180,13716.458
4,CA-2014-139892,10539.896
5,CA-2017-166709,10499.970
6,CA-2014-116904,9900.190
7,CA-2016-117121,9892.740
8,US-2016-107440,9135.190
9,CA-2016-158841,8805.040


	Min Sales Order

In [432]:
min_order = (
    df.groupby('Order ID')['Sales']
    .sum()
    .sort_values(ascending=True)
    .head(10)
    .reset_index()
)
min_order.columns = ['Order ID', 'Total Sales']
min_order

,Order ID,Total Sales
0,CA-2017-124114,0.556
1,CA-2016-168361,0.836
2,CA-2014-112403,0.852
3,US-2014-152723,0.876
4,US-2017-100209,1.080
5,CA-2015-146829,1.112
6,CA-2014-112718,1.167
7,US-2017-162068,1.188
8,CA-2017-106691,1.248
9,CA-2017-165099,1.392


	Number of Loss-Making Orders

In [405]:
# group profit for each order
order_profit = (
    df.groupby('Order ID')['Profit']
    .sum()
    .reset_index()
)
#filter lose orders
loss_orders = order_profit[order_profit['Profit'] < 0]
# number of lode orders
number_of_loss_orders = loss_orders.shape[0]
number_of_loss_orders

1022

	Orders with Biggest Loss

In [406]:
order_profit = (
    df.groupby('Order ID')['Profit']
    .sum()
    .reset_index()
)
biggest_loss_orders = order_profit \
    .sort_values(by='Profit') \
    .head(10) \
    .reset_index(drop=True)
biggest_loss_orders

,Order ID,Profit
0,CA-2016-108196,-6892.3748
1,US-2017-168116,-3825.3394
2,CA-2014-169019,-3791.1634
3,CA-2017-134845,-3424.3546
4,US-2017-122714,-2929.4845
5,CA-2017-131254,-2330.2698
6,CA-2015-147830,-1980.3794
7,CA-2014-139892,-1878.7892
8,CA-2015-116638,-1862.3124
9,CA-2016-130946,-1790.2708


shipping mode preformance

In [434]:

df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True)
df['Shipping Days'] = (df['Ship Date'] - df['Order Date']).dt.days
ship_mode_performance = (
    df.groupby('Ship Mode')
    .agg({
        'Shipping Days': 'mean',
        'Profit': 'sum'
    })
    .reset_index()
)
ship_mode_performance

,Ship Mode,Shipping Days,Profit
0,First Class,2.182705,48969.8399
1,Same Day,0.044199,15891.7589
2,Second Class,3.238046,57446.6354
3,Standard Class,5.006535,164088.7875


**location analysis**

Highest Region by Sales and Profit

In [407]:
region_performance = (
    df.groupby('Region')
    .agg({
        'Sales': 'sum',
        'Profit': 'sum'
    })
    .reset_index()
)
region_performance

,Region,Sales,Profit
0,Central,501239.8908,39706.3625
1,East,678781.2400,91522.7800
2,South,391721.9050,46749.4303
3,West,725457.8245,108418.4489


	Top 10 Cities by sales and profit

In [408]:
top_10_cities = (
    df.groupby('City')
    .agg({
        'Sales': 'sum',
        'Profit': 'sum'
    })
    .sort_values(by='Sales', ascending=False)
    .head(10)
    .reset_index()
)
top_10_cities

,City,Sales,Profit
0,New York City,256368.1610,62036.9837
1,Los Angeles,175851.3410,30440.7579
2,Seattle,119540.7420,29156.0967
3,San Francisco,112669.0920,17507.3854
4,Philadelphia,109077.0130,-13837.7674
5,Houston,64504.7604,-10153.5485
6,Chicago,48539.5410,-6654.5688
7,San Diego,47521.0290,6377.1960
8,Jacksonville,44713.1830,-2323.8350
9,Springfield,43054.3420,6200.6974


each region with most product profit




In [439]:
regions = df['Region'].unique()

region_product_profit = {}

for region in regions:
    region_df = df[df['Region'] == region]

    region_product_profit[region] = (
        region_df.groupby('Product Name')['Profit']
        .sum()
        .reset_index()
        .sort_values(by='Profit', ascending=False)
    )

region_product_profit

{'South':                                           Product Name     Profit
 374  Fellowes Pb500 Electric Punch Plastic Comb Bin...  3812.9700
 510  Hp Designjet T520 Inkjet Large Format Printer ...  1854.9894
 464  Hewlett-Packard Deskjet 3050A All-In-One Color...  1459.2000
 463               Hewlett Packard Laserjet 3310 Copier  1439.9760
 249                 Cisco 9971 Ip Video Phone Charcoal  1416.8000
 ..                                                 ...        ...
 391          Gbc Docubind P400 Electric Binding System -1306.5504
 254  Cisco Telepresence System Ex90 Videoconferenci... -1811.0784
 396   Gbc Ibimaster 500 Manual Proclick Binding System -1978.5480
 246  Chromcraft Bull-Nose Wood Oval Conference Tabl... -2865.0960
 268          Cubify Cubex 3D Printer Triple Head Print -3839.9904
 
 [1042 rows x 2 columns],
 'West':                                            Product Name     Profit
 324               Canon Imageclass 2200 Advanced Copier  6719.9808
 526   Fellowes

Products profit ability across regions

In [440]:
product_region_profit = (
    df.groupby(['Region', 'Product Name'])['Profit']
    .sum()
    .reset_index()
    .sort_values(by=['Region', 'Profit'], ascending=[True, False])
)
product_region_profit

,Region,Product Name,Profit
278,Central,Canon Imageclass 2200 Advanced Copier,8399.9760
476,Central,Gbc Ibimaster 500 Manual Proclick Binding System,3804.9000
281,Central,Canon Pc1060 Personal Laser Copier,2302.9671
634,Central,Ibico Epk-21 Electric Binding System,1700.9910
599,Central,Honeywell Enviracaire Portable Hepa Air Cleane...,1289.7885
...,...,...,...
3892,West,"Atlantic Metals Mobile 4-Shelf Bookcases, Cust...",-491.7150
4418,West,Hon 2090 “Pillow Soft” Series Mid Back Swivel/...,-547.9110
4721,West,O'Sullivan 4-Shelf Bookcase In Odessa Pine,-802.0974
5251,West,Zebra Gk420T Direct Thermal/Thermal Transfer P...,-938.2800


**discount analysis**

	Checking the impact of discount on sales

In [443]:
discount_impact = (
    df.groupby('Discount')
    .agg({
        'Sales': 'sum',
    })
    .reset_index()
)
discount_impact

,Discount,Sales
0,0.00,1.087908e+06
1,0.10,5.436935e+04
2,0.15,2.755852e+04
3,0.20,7.645944e+05
4,0.30,1.032267e+05
5,0.32,1.449346e+04
6,0.40,1.164178e+05
7,0.45,5.484974e+03
8,0.50,5.891854e+04
9,0.60,6.644700e+03


Checking the impact of discount on profit

In [444]:
discount_impact = (
    df.groupby('Discount')
    .agg({
        'Profit': 'sum'
    })
    .reset_index()
)
discount_impact

,Discount,Profit
0,0.00,320987.6032
1,0.10,9029.1770
2,0.15,1418.9915
3,0.20,90337.3060
4,0.30,-10369.2774
5,0.32,-2391.1377
6,0.40,-23057.0504
7,0.45,-2493.1111
8,0.50,-20506.4281
9,0.60,-5944.6552


**Sales Trend Analysis**

The trend of sales and profit over time (monthly/year)

In [448]:

df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Year-Month'] = df['Order Date'].dt.to_period('M')
monthly_trend = (
    df.groupby('Year-Month')[['Sales', 'Profit']]
    .sum()
    .reset_index()
)
yearly_trend = (
    df.groupby('Year')[['Sales', 'Profit']]
    .sum()
    .reset_index()
)

monthly_detailed = (
    df.groupby(['Year', 'Month'])[['Sales', 'Profit']]
    .sum()
    .reset_index()
    .sort_values(by=['Year', 'Month'])
)
monthly_sales
yearly_sales
monthly_detailed

,Year,Month,Sales,Profit
0,2014,1,14236.8950,2450.1907
1,2014,2,4519.8920,862.3084
2,2014,3,55691.0090,498.7299
3,2014,4,28295.3450,3488.8352
4,2014,5,23648.2870,2738.7096
5,2014,6,34595.1276,4976.5244
6,2014,7,33946.3930,-841.4826
7,2014,8,27909.4685,5318.1050
8,2014,9,81777.3508,8328.0994
9,2014,10,31453.3930,3448.2573


	Best year in sales

In [417]:
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Year'] = df['Order Date'].dt.year

yearly_sales = (
    df.groupby('Year')['Sales']
    .sum()
    .reset_index()
)
best_year = yearly_sales.sort_values(by='Sales', ascending=False).head(1)
best_year

,Year,Sales
3,2017,733215.2552
